# Audio Input Processing for RAG
### Transcribe once, then it's a standard text pipeline

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-pinecone pinecone langchain-text-splitters openai python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "mmrag-nomic"
NAMESPACE = "jfk"

client = OpenAI()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
nomic_embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")  # requires `ollama pull nomic-embed-text` + a running Ollama server
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 0: Provide an audio file
`tests_jfk.flac` (the classic ~11s Whisper test clip) ships alongside this notebook and runs out of the box. Drop your own short audio file into this folder and repoint `AUDIO_PATH` to try something else.

In [3]:
AUDIO_PATH = "tests_jfk.flac"

## Step 1: Transcribe with Whisper
There's no LangChain wrapper for audio transcription here — this is the one step in the pipeline that goes straight through the plain `openai` client instead.

In [4]:
with open(AUDIO_PATH, "rb") as f:
    transcript = client.audio.transcriptions.create(model="whisper-1", file=f)

print(transcript.text)

And so my fellow Americans, ask not what your country can do for you, ask what you can do for your country.


## Step 2: Chunk the transcript
This particular clip is a single sentence (~11s), so it'll likely collapse to one chunk — that's expected, not a bug. The "Try it yourself" section below is where a longer file actually exercises multi-chunk retrieval.

In [5]:
chunks = splitter.create_documents([transcript.text], metadatas=[{"source": AUDIO_PATH}])
print(f"{len(chunks)} transcript chunk(s)")

1 transcript chunk(s)


## Step 3: Embed with nomic-embed-text and store in Pinecone

In [6]:
pc = Pinecone()
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,  # nomic-embed-text output size
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

vs = PineconeVectorStore.from_documents(chunks, embedding=nomic_embeddings, index_name=INDEX_NAME, namespace=NAMESPACE)
print(f"Upserted {len(chunks)} chunks into '{INDEX_NAME}/{NAMESPACE}'")

Upserted 1 chunks into 'mmrag-nomic/jfk'


## Step 4: Retrieve and generate

In [7]:
RAG_PROMPT = """Answer the question using only the following context.

Context:
{context}

Question: {question}
Answer:"""

def answer_from_index(question, k=4):
    hits = vs.similarity_search(question, k=k, namespace=NAMESPACE)
    context = "\n\n".join(h.page_content for h in hits)
    response = llm.invoke(RAG_PROMPT.format(context=context, question=question)).content.strip()
    return hits, response

question = "According to the speaker, what should you ask instead of what your country can do for you?"
hits, response = answer_from_index(question)
print("Answer:\n", response)

Answer:
 You should ask what you can do for your country.


## Wrap-up
Once transcribed, audio becomes a standard text-RAG problem — the same chunk/embed/retrieve/generate pipeline as any other document. The interesting engineering work happens upstream, at transcription, not in retrieval.

## Try it yourself
1. Swap `AUDIO_PATH` for a longer recording (a few minutes of speech) and confirm retrieval actually has to choose among multiple chunks.
2. Request `response_format="verbose_json"` from `client.audio.transcriptions.create` to get per-segment timestamps, and store them as chunk metadata (`start`, `end`) so retrieved answers can cite a timestamp.
3. Try a question this ~11s clip genuinely can't answer, and confirm the model says so instead of guessing.

**Cleanup:** this notebook creates the `mmrag-nomic` index (no other notebook shares it). Delete the namespace, or the whole index if done with it:
```python
pc.Index(INDEX_NAME).delete(delete_all=True, namespace=NAMESPACE)
```